### Imports

In [1]:
from ray_tracing_simulator_nnModules_grad import PrismMirror, Ray, Plane, ReflectingPlane, RefractingPlane, Camera, visualize_camera_configuration, closest_point, rotx, get_rot_mat
import matplotlib.pyplot as plt
import numpy as np  
import torch
import random
seed = 0
# Python random
random.seed(seed)
# NumPy random
np.random.seed(seed)
# PyTorch random
torch.manual_seed(seed)
import scipy.io as sio
import os
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader, random_split, Dataset
torch.autograd.set_detect_anomaly(True)
import datetime
import time
from arenas.prism_arenas import Arena_reprojection_loss_two_cameras_prism_grid_distances
from utils import euclidean_distance #NOTE to self: Maybe this is not needed. Just use the in-built function?
from torch.utils.tensorboard import SummaryWriter
import argparse
import yaml

### Rays

In [2]:
# Initialize a ray with origin at (0,0,0) and direction along the z-axis
ray = Ray(origin=torch.tensor([0., 0., 0.]).unsqueeze(-1),# (3, 1) : x, y, z coordinates for one ray
          direction=torch.tensor([0., 0., 1.]).unsqueeze(-1) # (3, 1) : direction vector for one ray
          )

# Visualize the ray
fig, ax = ray.visualize()

In [3]:
# Change the length of the ray
ray = Ray(origin=torch.tensor([0., 0., 0.]).unsqueeze(-1),# (3, 1) : x, y, z coordinates for one ray
          direction=torch.tensor([0., 0., 1.]).unsqueeze(-1) # (3, 1) : direction vector for one ray
          )

# Visualize the ray
fig, ax = ray.visualize()
ax.text(0, 0, 2, f"Length = {ray.t.item()} unit")

# Shift the ray along the x-axis by 1 unit and change its length 5 times
ray.origin[0, 0] += 1.0
ray.t = 5 * ray.t
fig, ax = ray.visualize(fig, ax)
ax.text(1, 0, 2, f"Length = {ray.t.item()} unit")
plt.show()

In [6]:
# Initialize 10 rays
# Initialize with origin and target in this example

num_rays = 10
ray = Ray(origin=torch.rand(3, num_rays), # (3, 10) : x, y, z coordinates for 10 rays
            target=torch.rand(3, num_rays)  # (3, 10) : target points for 10 rays
            )
fig, ax = ray.visualize(color_labels=True)
ax.set_aspect('equal')

### Reflecting Plane

In [7]:
datatype = torch.float64
plane_width = torch.tensor([2.0]).to(dtype=datatype)
plane_height = torch.tensor([4.0]).to(dtype=datatype)
axes = torch.tensor(
    [[1., 0., 0.], # Normal vector along x-axis
     [0., 1., 0.], # Horizontal vector along y-axis (this will define the width direction)
     [0., 0., 1.]] # Vertical vector along z-axis (this will define the height direction)
).to(dtype=datatype)
reflecting_plane = ReflectingPlane(
                            axes=axes,
                            a=plane_width,
                            b=plane_height,
                            center=torch.tensor([0., 0., 0.]).unsqueeze(-1).to(torch.float64) # Centered at (0, 0, 0)
                            )
fig, ax = reflecting_plane.visualize()
ax.set_aspect('equal')

In [8]:
# Make a ray reflect off the reflecting plane
incident_ray = Ray(
    origin=reflecting_plane.center + 3.,
    target=reflecting_plane.center,
)
incident_ray.t = 50 * incident_ray.t # Extend the ray to ensure it hits the plane


In [9]:
# Make the ray incident on the plane (this is a forward pass for the plane)
reflected_ray, intersection_penalty = reflecting_plane(incident_ray) # The second output is a penalty term if the ray does not intersect the plane within its bounds

In [10]:
# Visualize the reflecting plane and the incident and reflected rays
fig, ax = reflecting_plane.visualize(fig=None, ax=None)
fig, ax = incident_ray.visualize(fig, ax, color='blue')
fig, ax = reflected_ray.visualize(fig, ax, color='red')

## Refracting Plane

In [36]:
datatype = torch.float64
plane_width = torch.tensor([2.0]).to(dtype=datatype)
plane_height = torch.tensor([4.0]).to(dtype=datatype)
axes = torch.tensor(
    [[1., 0., 0.], # Normal vector along x-axis
     [0., 1., 0.], # Horizontal vector along y-axis (this will define the width direction)
     [0., 0., 1.]] # Vertical vector along z-axis (this will define the height direction)
).to(dtype=datatype)
refracting_plane = RefractingPlane(
    axes=axes,
    a=plane_width,
    b=plane_height,
    center=torch.tensor([0., 0., 0.]).unsqueeze(-1).to(torch.float64), # Centered at (0, 0, 0)
    refractive_idx_1=1.0, # On the side facing the normal
    refractive_idx_2=1.66, # On the opposite side
)

# Create a ray that will refract through the refracting plane

incident_ray = Ray(
    origin=reflecting_plane.center + 3.,
    target=reflecting_plane.center,
)
incident_ray.t = 50 * incident_ray.t # Extend the ray to ensure it hits the plane

refracting_ray, intersection_penalty = refracting_plane(incident_ray) # The second output is a penalty term if the ray does not intersect the plane within its bounds
refracting_ray.t = 20.0 * refracting_ray.t # Extend the refracted ray for better visualization

In [37]:
# Visualize the refracting plane and the incident and refracted rays
# NOTE: Total internal reflection is not simulated by this library. Rays that undergo tir have a direction of (0,0,0). TIR handling can be implemented easily, but is not necessary for calibration
fig, ax = refracting_plane.visualize(fig=None, ax=None)
fig, ax = incident_ray.visualize(fig, ax, color='blue')
fig, ax = refracting_ray.visualize(fig, ax, color='red')
ax.set_aspect('equal')

/groups/branson/bransonlab/aniket/fly_walk_imaging/calibration_code/refraction_model/calprism/ray_tracing_simulator_nnModules_grad.py:602: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(sampled_points[0],


### Prism-Mirror

In [52]:
# Initialize a prism mirror and visualize it
prism_mirror = PrismMirror(prism_size=[1.,1.,1.],
                prism_rotation_6d=None,
                prism_center=[0.,0.,0.],
                refractive_index_glass=1.5,
                refractive_index_air=1.,
)
fig, ax = prism_mirror.visualize_prism()
ax.set_aspect('equal')


/groups/branson/bransonlab/aniket/fly_walk_imaging/calibration_code/refraction_model/calprism/ray_tracing_simulator_nnModules_grad.py:602: UserWarning: *c* argument looks like a single numeric RGB or RGBA sequence, which should be avoided as value-mapping will have precedence in case its length matches with *x* & *y*.  Please use the *color* keyword-argument or provide a 2D array with a single row if you intend to specify the same RGB or RGBA value for all points.
  ax.scatter(sampled_points[0],


In [53]:
# Create a ray that will interact with the prism mirror
incident_ray = Ray(
    origin=torch.tensor([[0.5], [0.1], [0.0]]).to(datatype), # Start above the prism
    target=torch.tensor([[0.0], [0.0], [0.0]]).to(datatype)  # Pointing towards the center of the prism
)
intermediate_ray_1, intermediate_ray2, emerging_ray, intersection_penalty = prism_mirror(incident_ray)
fig, ax = prism_mirror.visualize_prism()
fig, ax = incident_ray.visualize(fig, ax, color='blue')
fig, ax = intermediate_ray_1.visualize(fig, ax, color='green')
fig, ax = intermediate_ray2.visualize(fig, ax, color='orange')
fig, ax = emerging_ray.visualize(fig, ax, color='red')
ax.set_aspect('equal')

### Camera

To be implemented